# Data Prep from Silver to Gold

Requires silver parquet data

In gold layer we add features (especially spatial features like hexagon) and aggregate data from different datasets, etc.

In [1]:
import pandas as pd
import duckdb
import polars as pl
import numpy as np
import geopandas as gpd
from shapely import wkt
import h3
from shapely import from_wkb, from_wkt
from shapely.geometry.base import BaseGeometry
from pathlib import Path
import holidays


from datetime import datetime
import math

from run_config import PATHS, START_DATE, END_DATE

BRONZE_CENSUS_TRACTS = PATHS.bronze_census_tracts
BRONZE_COMMUNITY_AREA = PATHS.bronze_community_areas

SILVER_CENSUS_TRACTS = PATHS.silver_census_tracts
SILVER_COMMUNITY_AREA = PATHS.silver_community_areas
BRONZE_POIS = PATHS.bronze_osm_geo

SILVER_TAXI_PATH = PATHS.silver_taxi_trips
SILVER_WEATHER_PATH = PATHS.silver_weatherdata
SILVER_HEXAGON_PATH = PATHS.silver_hexagon

GOLD_TAXI_PATH = PATHS.gold_taxi_trips
GOLD_WEATHER_PATH = PATHS.gold_weatherdata
GOLD_HOURLY_DEMAND_HEXAGON = PATHS.gold_hourly_demand_hexagon
GOLD_HOURLY_DEMAND_CENSUS_TRACT = PATHS.gold_hourly_demand_census_tracts
GOLD_HOURLY_DEMAND_COMMUNITY_AREA = PATHS.gold_hourly_demand_community_areas

# Backward-compatible default plus supported comparison resolutions/bins
H3_RESOLUTION = 8
SUPPORTED_H3_RESOLUTIONS = (7, 8, 9)
SUPPORTED_TEMPORAL_BINS = {"hourly": "1h", "4hourly": "4h", "daily": "1d"}

# Weather related
START_TS = START_DATE
END_TS = END_DATE

Nachdem in silver alle Duplikate bereinigt wurden, können nun trip_id und taxi_id gedropped werden

## Taxi Data

### Adding hexagon

First we add hexagon data and then compare the hexagons of trips against the city boundaries of chicago. We only want trips with pickup hexagon in the city boundaries

In [2]:
def h3_from_row(row, lat_col: str, lon_col: str, resolution: int):
    """Return an H3 cell or None when either released centroid is missing."""
    lat, lon = row[lat_col], row[lon_col]
    if lat is None or lon is None:
        return None
    return h3.latlng_to_cell(lat, lon, resolution)


def h3_expr(lat_col: str, lon_col: str, resolution: int, alias: str) -> pl.Expr:
    return (
        pl.struct([lat_col, lon_col])
        .map_elements(
            lambda row: h3_from_row(row, lat_col, lon_col, resolution),
            return_dtype=pl.String,
        )
        .alias(alias)
    )


taxi_with_h3 = (
    pl.scan_parquet(SILVER_TAXI_PATH)
    .with_columns([
        h3_expr("pickup_centroid_latitude", "pickup_centroid_longitude", H3_RESOLUTION, "pickup_h3_cell"),
        h3_expr("dropoff_centroid_latitude", "dropoff_centroid_longitude", H3_RESOLUTION, "dropoff_h3_cell"),
    ])
)

# Check if there are trips without hexagon in Chicagos boundaries
valid_chicago_h3_cells = (
    pl.scan_parquet(SILVER_HEXAGON_PATH)
    .select("h3_cell")
)

count_before = (
    taxi_with_h3
    .select(pl.len().alias("n_rows_before"))
    .collect()
)

taxi_with_h3_chicago_only = (
    taxi_with_h3
    .join(
        valid_chicago_h3_cells,
        left_on="pickup_h3_cell",
        right_on="h3_cell",
        how="inner",
    )
)

count_after = (
    taxi_with_h3_chicago_only
    .select(pl.len().alias("n_rows_after"))
    .collect()
)

print(count_before)
print(count_after)

dropoff_h3_quality = (
    taxi_with_h3_chicago_only
    .select([
        pl.len().alias("trips"),
        pl.col("dropoff_h3_cell").is_null().sum().alias("missing_dropoff_h3"),
    ])
    .with_columns(
        (100 * pl.col("missing_dropoff_h3") / pl.col("trips")).alias("missing_dropoff_h3_pct")
    )
    .collect()
)
print(dropoff_h3_quality)

/var/folders/dy/nclg91xj2hs9nnr3bq4677q40000gn/T/ipykernel_15702/3535354708.py:53: UserWarning: Extension type 'geoarrow.wkb' is not registered; loading as its storage type.

To avoid this warning, register the extension type or set environment variable 'POLARS_UNKNOWN_EXTENSION_TYPE_BEHAVIOR' to 'load_as_storage' or 'load_as_extension'.

In Polars 2.0, the default behavior will change to 'load_as_extension'.
  .collect()


shape: (1, 1)
┌───────────────┐
│ n_rows_before │
│ ---           │
│ u32           │
╞═══════════════╡
│ 13387587      │
└───────────────┘
shape: (1, 1)
┌──────────────┐
│ n_rows_after │
│ ---          │
│ u32          │
╞══════════════╡
│ 13387586     │
└──────────────┘


shape: (1, 3)
┌──────────┬────────────────────┬────────────────────────┐
│ trips    ┆ missing_dropoff_h3 ┆ missing_dropoff_h3_pct │
│ ---      ┆ ---                ┆ ---                    │
│ u32      ┆ u32                ┆ f64                    │
╞══════════╪════════════════════╪════════════════════════╡
│ 13387586 ┆ 940428             ┆ 7.024627               │
└──────────┴────────────────────┴────────────────────────┘


### Save gold version


In [3]:
taxi_with_h3_chicago_only.sink_parquet(GOLD_TAXI_PATH)

print(f"Silver taxi parquet written to: {GOLD_TAXI_PATH}")

Silver taxi parquet written to: /Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/processed_data/gold_taxi.parquet


## Weather Data

Create hourly indexed dataset for weather data from 01.01.2024 to 24.05.2026

In [4]:
def mode_or_na(series: pd.Series):
    """Return most frequent non-null value, otherwise NA."""
    mode_values = series.dropna().mode()
    if len(mode_values) == 0:
        return pd.NA
    return mode_values.iloc[0]


def create_hourly_weather_gold(
    df: pd.DataFrame,
    start_ts: str = START_TS,
    end_ts: str = END_TS,
) -> pd.DataFrame:
    """
    Create hourly gold weather dataset.

    Steps:
    - Parse valid timestamp
    - Restrict to requested date range
    - Aggregate observations to hourly level
    - Create complete hourly timestamp spine
    - Reindex to all hours
    - Linearly interpolate numeric weather columns
    - Fill categorical/context columns
    - Add date/hour helper columns
    """

    df = df.copy()

    # 1. Basic cleanup
    df["valid"] = pd.to_datetime(df["valid"], utc=True)
    df = df.dropna(subset=["valid"])
    
    # Convert UTC to chicago timezone
    df["valid"] = df["valid"].dt.tz_convert("America/Chicago")

    start_ts = (
        pd.Timestamp(start_ts)
        .tz_localize(
            "America/Chicago",
            ambiguous=False, # takes winter time hour
            nonexistent="shift_forward"
        )
    )

    end_ts = (
        pd.Timestamp(end_ts)
        .tz_localize(
            "America/Chicago",
            ambiguous=False, # takes winter time hour
            nonexistent="shift_forward"
        )
    )

    # Optional: keep only relevant date range
    df = df[(df["valid"] >= start_ts) & (df["valid"] <= end_ts)]

    # 2. Create hourly timestamp
    df["valid_hour"] = df["valid"].dt.floor("h", ambiguous=False, nonexistent="shift_forward")

    # 3. Cast numeric columns
    numeric_cols = [
        "tmpc",   # temperature Celsius
        "relh",   # relative humidity
        "sknt",   # wind speed in knots
        "p01m",   # precipitation
        "vsby",   # visibility
        "lat",
        "lon",
    ]

    existing_numeric_cols = [col for col in numeric_cols if col in df.columns]

    for col in existing_numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # 4. Define aggregation rules
    agg_dict = {}

    # Numeric weather values: hourly mean
    for col in ["tmpc", "relh", "sknt", "vsby"]:
        if col in df.columns:
            agg_dict[col] = "mean"

    # Precipitation: hourly sum is usually more sensible than mean
    if "p01m" in df.columns:
        agg_dict["p01m"] = "sum"

    # Station/location fields
    if "station" in df.columns:
        agg_dict["station"] = mode_or_na

    if "lat" in df.columns:
        agg_dict["lat"] = "mean"

    if "lon" in df.columns:
        agg_dict["lon"] = "mean"

    # Categorical weather condition
    if "skyc1" in df.columns:
        agg_dict["skyc1"] = mode_or_na

    # 5. Aggregate to hourly level
    hourly = (
        df
        .groupby("valid_hour", as_index=True)
        .agg(agg_dict)
        .sort_index()
    )

    # 6. Complete hourly index from start to end
    full_hourly_index = pd.date_range(
        start=start_ts,
        end=end_ts,
        freq="h",
        name="valid_hour",
    )

    hourly = hourly.reindex(full_hourly_index)

    # 7. Interpolate numeric columns over missing hours
    numeric_interpolate_cols = [
        col for col in ["tmpc", "relh", "sknt", "p01m", "vsby", "lat", "lon"]
        if col in hourly.columns
    ]

    hourly[numeric_interpolate_cols] = (
        hourly[numeric_interpolate_cols]
        .interpolate(method="time", limit_direction="both")
    )

    # 8. Fill categorical/context columns
    categorical_fill_cols = [
        col for col in ["station", "skyc1"]
        if col in hourly.columns
    ]

    for col in categorical_fill_cols:
        hourly[col] = hourly[col].ffill().bfill()

    # 9. Back to normal dataframe
    gold = hourly.reset_index()
    
    # Onehot encoding
    skyc1_dummies = pd.get_dummies(
        gold["skyc1"],
        prefix="skyc1",
        dummy_na=False,
        dtype="int8",
    )

    gold = pd.concat([gold, skyc1_dummies], axis=1)
    
    # Drop unused columns
    gold = gold.drop(columns=["skyc1", "station", "lat", "lon"])

    return gold

weather_silver = pd.read_parquet(SILVER_WEATHER_PATH)

weather_gold = create_hourly_weather_gold(
    weather_silver,
    start_ts=START_TS,
    end_ts=END_TS,
)

weather_gold.to_parquet(
    GOLD_WEATHER_PATH,
    index=False,
    engine="pyarrow",
    compression="snappy",
)

print(f"Gold weather parquet written to: {GOLD_WEATHER_PATH}")
print(f"Rows: {len(weather_gold):,}")
print(f"Start: {weather_gold['valid_hour'].min()}")
print(f"End: {weather_gold['valid_hour'].max()}")
print()
print(weather_gold.head().to_string(index=False))
print()
print(weather_gold.tail().to_string(index=False))

Gold weather parquet written to: /Users/moritz/Documents/Master Studium/Advanced Analytics and Application/assignment/Group-3-AAA/data/processed_data/gold_weather.parquet
Rows: 20,424
Start: 2024-01-01 00:00:00-06:00
End: 2026-05-01 00:00:00-05:00

               valid_hour  tmpc  relh  sknt  vsby   p01m  skyc1_BKN  skyc1_CLR  skyc1_FEW  skyc1_OVC  skyc1_SCT  skyc1_VV 
2024-01-01 00:00:00-06:00  1.11 75.26  12.0  10.0 0.0001          0          0          0          1          0          0
2024-01-01 01:00:00-06:00  1.11 75.26  12.0   8.0 0.0001          0          0          0          1          0          0
2024-01-01 02:00:00-06:00  0.56 81.63  10.0   7.0 0.0001          0          0          0          1          0          0
2024-01-01 03:00:00-06:00  0.56 78.34  13.0   9.0 0.0001          0          0          0          1          0          0
2024-01-01 04:00:00-06:00  0.56 75.17  11.5  10.0 0.0002          0          0          0          1          0          0

            

## Points of Interest

In [5]:
pois = gpd.read_parquet(BRONZE_POIS)

print("\nPOIs:")
print(pois.shape)
print(pois.columns)
print(pois.crs)
print(type(pois.geometry.iloc[0]))



POIs:
(9115, 21)
Index(['poi_category', 'name', 'amenity', 'shop', 'railway', 'tourism',
       'historic', 'man_made', 'brand', 'operator', 'opening_hours',
       'addr_housenumber', 'addr_street', 'addr_city', 'website', 'phone',
       'geom_type_original', 'area_m2', 'lat', 'lon', 'geometry'],
      dtype='str')
{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy"

### Hexagon

In [6]:
hexagons = gpd.read_parquet(SILVER_HEXAGON_PATH)

print("Hexagons:")
print(hexagons.shape)
print(hexagons.columns)
print(hexagons.crs)
print(type(hexagons.geometry.iloc[0]))

# CRS prüfen
print("Hexagon CRS:", hexagons.crs)
print("POI CRS:", pois.crs)

# Falls CRS unterschiedlich sind: POIs auf CRS der Hexagons bringen
if pois.crs != hexagons.crs:
    pois = pois.to_crs(hexagons.crs)

print("CRS identisch:", pois.crs == hexagons.crs)

pois_with_h3 = gpd.sjoin(
    pois,
    hexagons[["h3_cell", "geometry"]],
    how="left",
    predicate="within"
)

pois_with_h3.head()
# technische Join-Spalte entfernen
pois_with_h3 = pois_with_h3.drop(columns=["index_right"])

# prüfen, wie viele POIs keine h3_cell bekommen haben
missing_h3 = pois_with_h3["h3_cell"].isna().sum()

print("POIs gesamt:", len(pois_with_h3))
print("POIs ohne h3_cell:", missing_h3)
print("Anteil ohne h3_cell:", round(missing_h3 / len(pois_with_h3) * 100, 2), "%")

pois_with_h3[["name", "poi_category", "lat", "lon", "h3_cell"]].head()

print("POIs vorher:", len(pois))
print("POIs nach Join:", len(pois_with_h3))

duplicate_rows = len(pois_with_h3) - len(pois)

print("Zusätzliche Zeilen durch Join:", duplicate_rows)

Hexagons:
(853, 3)
Index(['h3_cell', 'h3_resolution', 'geometry'], dtype='str')
{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbr

### Census Tract

In [7]:
census_tracts = gpd.read_parquet(SILVER_CENSUS_TRACTS)

print("Census Tracts:")
print(census_tracts.shape)
print(census_tracts.columns)
print(census_tracts.crs)
print(type(census_tracts.geometry.iloc[0]))

# CRS prüfen
print("Census Tract CRS:", census_tracts.crs)
print("POI CRS:", pois.crs)

# Falls CRS unterschiedlich sind: POIs auf CRS der Hexagons bringen
if pois.crs != census_tracts.crs:
    pois = pois.to_crs(census_tracts.crs)

print("CRS identisch:", pois.crs == census_tracts.crs)

pois_with_census_tract = gpd.sjoin(
    pois_with_h3,
    census_tracts[["CENSUS_T_1", "geometry"]],
    how="left",
    predicate="within"
)

pois_with_census_tract.head()
# technische Join-Spalte entfernen
pois_with_census_tract = pois_with_census_tract.drop(columns=["index_right"])

# prüfen, wie viele POIs keine h3_cell bekommen haben
missing_census_tract = pois_with_census_tract["CENSUS_T_1"].isna().sum()

print("POIs gesamt:", len(pois_with_census_tract))
print("POIs ohne census tract:", missing_census_tract)
print("Anteil ohne census tract", round(missing_census_tract / len(pois_with_census_tract) * 100, 2), "%")

pois_with_census_tract[["name", "poi_category", "lat", "lon", "CENSUS_T_1"]].head()

print("POIs vorher:", len(pois))
print("POIs nach Join:", len(pois_with_census_tract))

duplicate_rows = len(pois_with_census_tract) - len(pois)

print("Zusätzliche Zeilen durch Join:", duplicate_rows)

Census Tracts:
(878, 18)
Index(['OBJECTID', 'CENSUS_TRA', 'CENSUS_T_1', 'TRACT_FIPS', 'TRACT_CENT',
       'TRACT_CE_1', 'TRACT_CE_2', 'TRACT_CE_3', 'TRACT_COMM', 'TRACT_NUMA',
       'TRACT_CENS', 'PERIMETER', 'DATA_ADMIN', 'TRACT_CREA', 'TRACT_CR_1',
       'SHAPE_AREA', 'SHAPE_LEN', 'geometry'],
      dtype='str')
{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy":

### Community Area

In [8]:
community_area = gpd.read_parquet(SILVER_COMMUNITY_AREA)

print("Community Area:")
print(community_area.shape)
print(community_area.columns)
print(community_area.crs)
print(type(community_area.geometry.iloc[0]))

# CRS prüfen
print("Community Area CRS:", community_area.crs)
print("POI CRS:", pois.crs)

# Falls CRS unterschiedlich sind: POIs auf CRS der Hexagons bringen
if pois.crs != community_area.crs:
    pois = pois.to_crs(community_area.crs)

print("CRS identisch:", pois.crs == community_area.crs)

pois_with_community_area = gpd.sjoin(
    pois_with_census_tract,
    community_area[["AREA_NUM_1", "geometry"]],
    how="left",
    predicate="within"
)

pois_with_community_area.head()
# technische Join-Spalte entfernen
pois_with_community_area = pois_with_community_area.drop(columns=["index_right"])

# prüfen, wie viele POIs keine h3_cell bekommen haben
missing_community_area = pois_with_community_area["AREA_NUM_1"].isna().sum()

print("POIs gesamt:", len(pois_with_community_area))
print("POIs ohne community area:", missing_community_area)
print("Anteil ohne community area", round(missing_community_area / len(pois_with_community_area) * 100, 2), "%")

pois_with_community_area[["name", "poi_category", "lat", "lon", "AREA_NUM_1"]].head()

print("POIs vorher:", len(pois))
print("POIs nach Join:", len(pois_with_community_area))

duplicate_rows = len(pois_with_community_area) - len(pois)

print("Zusätzliche Zeilen durch Join:", duplicate_rows)

Community Area:
(77, 6)
Index(['AREA_NUMBE', 'COMMUNITY', 'AREA_NUM_1', 'SHAPE_AREA', 'SHAPE_LEN',
       'geometry'],
      dtype='str')
{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north

In [9]:
# Drop all missing h3_cell, CENSUS_T_1 and AREA_NUM_1
pois_with_spatials = pois_with_community_area.dropna(subset=["h3_cell", "CENSUS_T_1", "AREA_NUM_1"])

def group_pois_by_spatial(spatial_col):
    
    poi_h3_categories = pois_with_spatials[[spatial_col, "poi_category"]].copy()

    poi_h3_categories.head()
    
    poi_counts_long = (
    poi_h3_categories
        .groupby([spatial_col, "poi_category"])
        .size()
        .reset_index(name="poi_count")
    )

    poi_counts_long.head()

    poi_counts_wide = (
        poi_counts_long
        .pivot_table(
            index=spatial_col,
            columns="poi_category",
            values="poi_count",
            fill_value=0
        )
        .reset_index()
    )

    poi_counts_wide.head()
    
    poi_counts_wide.columns.name = None

    poi_counts_wide.head()
    
    return poi_counts_wide

pois_grouped_hexagon = group_pois_by_spatial("h3_cell")
pois_grouped_census_tract = group_pois_by_spatial("CENSUS_T_1")
pois_grouped_community_area = group_pois_by_spatial("AREA_NUM_1")

## Hourly Dataset

### Preparations

Here we create an hourly dataset, whereby each hour contains a row for each hexagon. 

We use cyclic encoding for columns like hour, weekday or month to be aware of the distance between time instances (e. g. hour 23 -> 0).

And we merge with the weather data.

These steps are done before the cross join as they are only dependend on time and not on the spatial unit

In [10]:
START = datetime.fromisoformat(START_TS).date()
END = datetime.fromisoformat(END_TS).date()

il_holidays = holidays.country_holidays(
    country="US",
    subdiv="IL", # Illinois = relevant for Chicago
    years=[2024, 2025, 2026]
)

holiday_df = (
    pl.DataFrame([
        {
            "date": d,
            "is_holiday": 1,
        }
        for d, name in sorted(il_holidays.items())
        if START <= d <= END
    ])
    .with_columns(
        pl.col("date").cast(pl.Date),
        pl.col("is_holiday").cast(pl.Int8),
    )
)

In [11]:
CHICAGO_TZ = "America/Chicago"

# --------------------------------------------------
# 1. Lokale Stunden erzeugen, aber timezone-naive
# --------------------------------------------------

hourly_index = pd.date_range(
    start=START_TS,
    end=END_TS,
    freq="h",
    inclusive="both",
    name="datetime_hour",
)

hours_pd = pd.DataFrame({
    "datetime_hour": hourly_index
})

hours = (
    pl.from_pandas(hours_pd)
    .with_columns(
        pl.col("datetime_hour")
        .cast(pl.Datetime("us"))
        .dt.truncate("1h")
        .alias("datetime_hour")
    )
)


# --------------------------------------------------
# 2. Zeitfeatures vor dem Cross Join hinzufügen
# --------------------------------------------------

hours_with_features = (
    hours
    .with_columns([
        pl.col("datetime_hour").dt.month().alias("month"),
        pl.col("datetime_hour").dt.weekday().alias("weekday"),
        pl.col("datetime_hour").dt.hour().alias("hour"),
    ])
    .with_columns([
        ((2 * math.pi * (pl.col("month") - 1) / 12).sin()).alias("month_sin"),
        ((2 * math.pi * (pl.col("month") - 1) / 12).cos()).alias("month_cos"),

        ((2 * math.pi * (pl.col("weekday") - 1) / 7).sin()).alias("weekday_sin"),
        ((2 * math.pi * (pl.col("weekday") - 1) / 7).cos()).alias("weekday_cos"),

        ((2 * math.pi * pl.col("hour") / 24).sin()).alias("hour_sin"),
        ((2 * math.pi * pl.col("hour") / 24).cos()).alias("hour_cos"),
    ])
)


# --------------------------------------------------
# 3. Wetterdaten auf denselben Join-Key bringen
# --------------------------------------------------
weather_hourly = (
    pl.scan_parquet(GOLD_WEATHER_PATH)
    .with_columns(
        pl.col("valid_hour")
        .dt.convert_time_zone(CHICAGO_TZ)
        .dt.truncate("1h")
        .dt.replace_time_zone(None)   # wichtig: timezone-naive machen
        .cast(pl.Datetime("us"))      # exakt gleicher Typ wie hours
        .alias("datetime_hour")
    )
    .group_by("datetime_hour")
    .agg(
        pl.all()
        .exclude(["valid_hour", "datetime_hour"])
        .first()
    )
)


# --------------------------------------------------
# 4. Weather vor dem Cross Join an Hours joinen
# --------------------------------------------------

hours_with_weather = (
    hours_with_features
    .lazy()
    .join(
        weather_hourly,
        on="datetime_hour",
        how="left",
    )
).lazy()

hours_with_weather = (
    hours_with_weather
    .filter(
        pl.col("tmpc").is_not_null() # 3 null values für 10.03.24, 08.03.26 and 09.03.25 due to winter -> sommer time
    )
)

# Add holidays
hours_with_holidays = (
    hours_with_weather
    .with_columns(
        pl.col("datetime_hour").dt.date().alias("date")
    )
    .join(
        holiday_df.lazy(),
        on="date",
        how="left"
    )
    .with_columns(
        pl.col("is_holiday").fill_null(0).cast(pl.Int8),
    )
).lazy()

In [12]:
hours_with_holidays.collect_schema()

Schema([('datetime_hour', Datetime(time_unit='us', time_zone=None)),
        ('month', Int8),
        ('weekday', Int8),
        ('hour', Int8),
        ('month_sin', Float64),
        ('month_cos', Float64),
        ('weekday_sin', Float64),
        ('weekday_cos', Float64),
        ('hour_sin', Float64),
        ('hour_cos', Float64),
        ('tmpc', Float64),
        ('relh', Float64),
        ('sknt', Float64),
        ('vsby', Float64),
        ('p01m', Float64),
        ('skyc1_BKN', Int8),
        ('skyc1_CLR', Int8),
        ('skyc1_FEW', Int8),
        ('skyc1_OVC', Int8),
        ('skyc1_SCT', Int8),
        ('skyc1_VV ', Int8),
        ('date', Date),
        ('is_holiday', Int8)])

### Hexagon Hourly

In [13]:
hexagon = (
    pl.scan_parquet(SILVER_HEXAGON_PATH)
    .select([
        pl.col("h3_cell").alias("h3_cell"),  
    ])
    .unique()
    .collect()
).lazy()

hexagon_with_pois = (
    hexagon
    .join(
        pl.from_pandas(pois_grouped_hexagon).lazy(),
        on="h3_cell",
        how="left"
    )
)

hexagon_hourly = (
    hours_with_holidays
    .join(
        hexagon_with_pois,
        how="cross",
    )
)

hexagon_hourly.sink_parquet(GOLD_HOURLY_DEMAND_HEXAGON)

In [14]:
duckdb.sql(f"""
    SELECT count(*)
    FROM  read_parquet('{GOLD_HOURLY_DEMAND_HEXAGON}')      
           """).show(max_rows=100)

duckdb.sql(f"""
    SELECT EXTRACT('hour' FROM datetime_hour), count(*)
    FROM  read_parquet('{GOLD_HOURLY_DEMAND_HEXAGON}')
    GROUP BY EXTRACT('hour' FROM datetime_hour)
    ORDER BY EXTRACT('hour' FROM datetime_hour) asc       
           """).show(max_rows=100)



┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     17419966 │
└──────────────┘

┌───────────────────────────────────────┬──────────────┐
│ main.date_part('hour', datetime_hour) │ count_star() │
│                 int64                 │    int64     │
├───────────────────────────────────────┼──────────────┤
│                                     0 │       726756 │
│                                     1 │       725903 │
│                                     2 │       723344 │
│                                     3 │       725903 │
│                                     4 │       725903 │
│                                     5 │       725903 │
│                                     6 │       725903 │
│                                     7 │       725903 │
│                                     8 │       725903 │
│                                     9 │       725903 │
│                                    10 │       725903 │
│                                    11 │ 

### Census Tract Hourly

In [15]:
census_tracts = (
    pl.scan_parquet(SILVER_CENSUS_TRACTS)
    .select([
        pl.col("CENSUS_T_1").alias("census_tract"),  
    ])
    .unique()
    .collect()
).lazy()

census_tracts_with_pois = (
    census_tracts
    .join(
        pl.from_pandas(pois_grouped_census_tract).lazy(),
        left_on="census_tract",
        right_on="CENSUS_T_1",
        how="left"
    )
)

census_tract_hourly = (
    hours_with_holidays
    .join(
        census_tracts_with_pois,
        how="cross",
    )
)

census_tracts_with_pois.sink_parquet(GOLD_HOURLY_DEMAND_CENSUS_TRACT)

In [16]:
n_hours = (
    hours_with_holidays
    .select(pl.len().alias("n_rows"))
    .collect()
    .item()
)
n_census_tracts = (
    census_tracts
    .select(pl.len().alias("n_rows"))
    .collect()
    .item()
)
actual_rows = duckdb.sql(f"""
SELECT
    COUNT(*) AS n_rows,
FROM read_parquet('{GOLD_HOURLY_DEMAND_CENSUS_TRACT}')
""").df()["n_rows"].iloc[0]

print(n_hours * n_census_tracts)
print(actual_rows)

17930516
878


### Community Area Hourly

In [17]:
community_area = (
    pl.scan_parquet(SILVER_COMMUNITY_AREA)
    .select(
        pl.col("AREA_NUM_1").alias("community_area")
    )
    .unique()
    .collect()
).lazy()

community_area_with_pois = (
    community_area
    .join(
        pl.from_pandas(pois_grouped_community_area).lazy(),
        left_on="community_area",
        right_on="AREA_NUM_1",
        how="left"
    )
)

community_area_hourly = (
    hours_with_holidays
    .join(
        community_area_with_pois,
        how="cross",
    )
)

community_area_with_pois.sink_parquet(GOLD_HOURLY_DEMAND_COMMUNITY_AREA)

In [18]:
n_hours = ( 
    hours_with_holidays
    .select(pl.len().alias("n_rows"))
    .collect()
    .item()
)
n_community_areas = (
    community_area
    .select(pl.len().alias("n_rows"))
    .collect()
    .item()
)
actual_rows = duckdb.sql(f"""
SELECT
    COUNT(*) AS n_rows,
FROM read_parquet('{GOLD_HOURLY_DEMAND_COMMUNITY_AREA}')
""").df()["n_rows"].iloc[0]

print(n_hours * n_community_areas)
print(actual_rows)

1572494
77


In [19]:
duckdb.sql(f"""
SELECT
    *,
FROM read_parquet('{GOLD_HOURLY_DEMAND_COMMUNITY_AREA}')
""").show()

┌────────────────┬────────────┬──────────┬────────┬───────────────┐
│ community_area │ food_drink │ landmark │  shop  │ train_station │
│    varchar     │   double   │  double  │ double │    double     │
├────────────────┼────────────┼──────────┼────────┼───────────────┤
│ 23             │       35.0 │      7.0 │   30.0 │           0.0 │
│ 69             │       23.0 │     20.0 │   25.0 │           6.0 │
│ 19             │       63.0 │      2.0 │   45.0 │           0.0 │
│ 28             │      495.0 │     41.0 │   78.0 │          36.0 │
│ 1              │       88.0 │     11.0 │   30.0 │          14.0 │
│ 35             │       23.0 │     90.0 │    6.0 │           5.0 │
│ 50             │       11.0 │      4.0 │    6.0 │           0.0 │
│ 4              │      116.0 │     10.0 │   22.0 │          11.0 │
│ 59             │       20.0 │      3.0 │   11.0 │           4.0 │
│ 43             │       42.0 │      6.0 │   25.0 │           5.0 │
│ ·              │         ·  │       ·  │     ·

### Function to aggregate taxi data into the spatio-temporal datasets

In [20]:
def add_taxi_data(
    skeleton: pl.LazyFrame | pl.DataFrame,
    taxi: pl.LazyFrame | pl.DataFrame,
    spatial_col: str,
    taxi_spatial_col: str,
    timestamp_col: str = "trip_start_timestamp",
    datetime_col: str = "datetime_hour",
    temporal_bin: str = "1h",
) -> pl.LazyFrame:
    """
    Adds hourly trip-start demand to a complete hour × spatial-unit skeleton.

    Parameters
    ----------
    skeleton:
        Complete base dataset with one row per datetime_col × spatial_col.
        Example: datetime_hour × h3_cell, datetime_hour × census_tract, etc.

    taxi:
        Taxi trip dataset with one row per trip.

    spatial_col:
        Spatial column name in the skeleton/output.
        Example: "h3_cell", "census_tract", "community_area".

    taxi_spatial_col:
        Spatial pickup column name in taxi.
        Example: "pickup_h3_cell", "pickup_census_tract", "pickup_community_area".

    timestamp_col:
        Taxi trip start timestamp column.

    datetime_col:
        Hourly timestamp column used in skeleton/output.

    demand_col:
        Output demand column name.

    Returns
    -------
    pl.LazyFrame
        Skeleton enriched with demand_col.
    """

    if isinstance(skeleton, pl.DataFrame):
        skeleton = skeleton.lazy()

    if isinstance(taxi, pl.DataFrame):
        taxi = taxi.lazy()

    taxi_demand = (
        taxi
        .filter(
            pl.col(timestamp_col).is_not_null()
            & pl.col(taxi_spatial_col).is_not_null() # TODO Census Tract contains nulls
        )
        .with_columns([
            pl.col(timestamp_col)
            .cast(pl.Datetime("us"))
            .dt.truncate(temporal_bin)
            .alias(datetime_col),

            pl.col(taxi_spatial_col).alias(spatial_col),
        ])
        .group_by([
            datetime_col,
            spatial_col,
        ])
        .agg(
            # aggregations:
            pl.len().alias('trip_count'),
            pl.col('trip_seconds').sum().alias('trip_seconds_sum'),
            pl.col('trip_seconds').mean().alias('trip_seconds_mean'),
            pl.col('trip_seconds').min().alias('trip_seconds_min'),
            pl.col('trip_seconds').max().alias('trip_seconds_max'),
            pl.col('trip_miles').sum().alias('trip_miles_sum'),
            pl.col('trip_miles').mean().alias('trip_miles_mean'),
            pl.col('trip_miles').min().alias('trip_miles_min'),
            pl.col('trip_miles').max().alias('trip_miles_max'),
            pl.col('fare').sum().alias('fare_sum'),
            pl.col('fare').mean().alias('fare_mean'),
            pl.col('fare').min().alias('fare_min'),
            pl.col('fare').max().alias('fare_max'),
            pl.col('tips').sum().alias('tips_sum'),
            pl.col('tips').mean().alias('tips_mean'),
            pl.col('tips').min().alias('tips_min'),
            pl.col('tips').max().alias('tips_max'),
            pl.col('tolls').sum().alias('tolls_sum'),
            pl.col('tolls').mean().alias('tolls_mean'),
            pl.col('tolls').min().alias('tolls_min'),
            pl.col('tolls').max().alias('tolls_max'),
            pl.col('extras').sum().alias('extras_sum'),
            pl.col('extras').mean().alias('extras_mean'),
            pl.col('extras').min().alias('extras_min'),
            pl.col('extras').max().alias('extras_max'),
            pl.col('trip_total').sum().alias('trip_total_sum'),
            pl.col('trip_total').mean().alias('trip_total_mean'),
            pl.col('trip_total').min().alias('trip_total_min'),
            pl.col('trip_total').max().alias('trip_total_max'),
            pl.col('payment_type').drop_nulls().mode().first().alias('most_common_payment_type')
        )    
    )

    result = (
        skeleton
        .with_columns(
            pl.col(datetime_col)
            .cast(pl.Datetime("us"))
            .dt.truncate(temporal_bin)
            .alias(datetime_col)
        )
        .join(
            taxi_demand,
            on=[datetime_col, spatial_col],
            how="left",
        )
        .with_columns([
            pl.col("trip_count").fill_null(0).alias("trip_count"),

            pl.col("trip_seconds_sum").fill_null(0).alias("trip_seconds_sum"),
            pl.col("trip_seconds_mean").fill_null(0).alias("trip_seconds_mean"),
            pl.col("trip_seconds_min").fill_null(0).alias("trip_seconds_min"),
            pl.col("trip_seconds_max").fill_null(0).alias("trip_seconds_max"),

            pl.col("trip_miles_sum").fill_null(0).alias("trip_miles_sum"),
            pl.col("trip_miles_mean").fill_null(0).alias("trip_miles_mean"),
            pl.col("trip_miles_min").fill_null(0).alias("trip_miles_min"),
            pl.col("trip_miles_max").fill_null(0).alias("trip_miles_max"),

            pl.col("fare_sum").fill_null(0).alias("fare_sum"),
            pl.col("fare_mean").fill_null(0).alias("fare_mean"),
            pl.col("fare_min").fill_null(0).alias("fare_min"),
            pl.col("fare_max").fill_null(0).alias("fare_max"),

            pl.col("tips_sum").fill_null(0).alias("tips_sum"),
            pl.col("tips_mean").fill_null(0).alias("tips_mean"),
            pl.col("tips_min").fill_null(0).alias("tips_min"),
            pl.col("tips_max").fill_null(0).alias("tips_max"),

            pl.col("tolls_sum").fill_null(0).alias("tolls_sum"),
            pl.col("tolls_mean").fill_null(0).alias("tolls_mean"),
            pl.col("tolls_min").fill_null(0).alias("tolls_min"),
            pl.col("tolls_max").fill_null(0).alias("tolls_max"),

            pl.col("extras_sum").fill_null(0).alias("extras_sum"),
            pl.col("extras_mean").fill_null(0).alias("extras_mean"),
            pl.col("extras_min").fill_null(0).alias("extras_min"),
            pl.col("extras_max").fill_null(0).alias("extras_max"),

            pl.col("trip_total_sum").fill_null(0).alias("trip_total_sum"),
            pl.col("trip_total_mean").fill_null(0).alias("trip_total_mean"),
            pl.col("trip_total_min").fill_null(0).alias("trip_total_min"),
            pl.col("trip_total_max").fill_null(0).alias("trip_total_max"),
        ])
        .with_columns(
            pl.col("most_common_payment_type")
            .fill_null("No trips")
            .alias("most_common_payment_type")
        )
    )

    return result

## Parameterized model-table builder

The legacy outputs remain H3 resolution 8 / hourly for backward compatibility. `build_parameterized_demand` additionally supports H3 resolutions 7, 8, and 9 or census tracts, combined with hourly, 4-hourly, or daily bins. Every output contains the complete time × spatial-unit skeleton, including zero-demand rows, which is necessary for unbiased demand prediction. Alternative outputs receive configuration-specific filenames and are generated only when explicitly requested because the fine hourly grids can be several GB.

In [21]:
WEATHER_FEATURES = [
    "tmpc", "relh", "sknt", "vsby", "p01m",
    "skyc1_BKN", "skyc1_CLR", "skyc1_FEW", "skyc1_OVC", "skyc1_SCT", "skyc1_VV ",
]
POI_FEATURES = ["food_drink", "landmark", "shop", "train_station"]


def make_time_context(temporal_bin: str) -> pl.LazyFrame:
    """Aggregate hourly context to the requested bin and recompute calendar encodings."""
    if temporal_bin not in SUPPORTED_TEMPORAL_BINS.values():
        raise ValueError(f"Unsupported temporal bin: {temporal_bin}")

    available = hours_with_holidays.collect_schema().names()
    weather = [col for col in WEATHER_FEATURES if col in available]
    context = (
        hours_with_holidays
        .with_columns(pl.col("datetime_hour").dt.truncate(temporal_bin))
        .group_by("datetime_hour")
        .agg([
            *[pl.col(col).mean().alias(col) for col in weather],
            pl.col("is_holiday").max().cast(pl.Int8).alias("is_holiday"),
        ])
        .with_columns([
            pl.col("datetime_hour").dt.date().alias("date"),
            pl.col("datetime_hour").dt.month().alias("month"),
            pl.col("datetime_hour").dt.weekday().alias("weekday"),
            pl.col("datetime_hour").dt.hour().alias("hour"),
        ])
        .with_columns([
            ((2 * math.pi * (pl.col("month") - 1) / 12).sin()).alias("month_sin"),
            ((2 * math.pi * (pl.col("month") - 1) / 12).cos()).alias("month_cos"),
            ((2 * math.pi * (pl.col("weekday") - 1) / 7).sin()).alias("weekday_sin"),
            ((2 * math.pi * (pl.col("weekday") - 1) / 7).cos()).alias("weekday_cos"),
            ((2 * math.pi * pl.col("hour") / 24).sin()).alias("hour_sin"),
            ((2 * math.pi * pl.col("hour") / 24).cos()).alias("hour_cos"),
        ])
    )
    return context


def make_h3_spatial_context(resolution: int) -> pl.LazyFrame:
    """Create a Chicago H3 grid at r7/r8/r9 and attach POI counts at that resolution."""
    if resolution not in SUPPORTED_H3_RESOLUTIONS:
        raise ValueError(f"Unsupported H3 resolution: {resolution}")

    base_r8 = (
        pl.read_parquet(SILVER_HEXAGON_PATH).get_column("h3_cell").drop_nulls().unique().to_list()
    )
    if resolution == 7:
        cells = sorted({h3.cell_to_parent(cell, 7) for cell in base_r8})
    elif resolution == 8:
        cells = sorted(base_r8)
    else:
        cells = sorted({child for cell in base_r8 for child in h3.cell_to_children(cell, 9)})

    poi_frame = pois[["lat", "lon", "poi_category"]].dropna(subset=["lat", "lon"]).copy()
    poi_frame["h3_cell"] = [h3.latlng_to_cell(lat, lon, resolution) for lat, lon in zip(poi_frame.lat, poi_frame.lon)]
    poi_counts = (
        poi_frame.groupby(["h3_cell", "poi_category"]).size().unstack(fill_value=0).reset_index()
    )
    for col in POI_FEATURES:
        if col not in poi_counts:
            poi_counts[col] = 0

    return (
        pl.DataFrame({"h3_cell": cells}).lazy()
        .join(pl.from_pandas(poi_counts[["h3_cell", *POI_FEATURES]]).lazy(), on="h3_cell", how="left")
        .with_columns([pl.col(col).fill_null(0) for col in POI_FEATURES])
        .with_columns(pl.lit(resolution).cast(pl.Int8).alias("h3_resolution"))
    )


def build_parameterized_demand(
    spatial_kind: str,
    temporal_label: str,
    h3_resolution: int | None = None,
    output_path: Path | None = None,
) -> tuple[pl.LazyFrame, Path]:
    """Build a zero-inclusive predictive table for one spatial/temporal configuration."""
    if temporal_label not in SUPPORTED_TEMPORAL_BINS:
        raise ValueError(f"temporal_label must be one of {tuple(SUPPORTED_TEMPORAL_BINS)}")
    temporal_bin = SUPPORTED_TEMPORAL_BINS[temporal_label]
    time_context = make_time_context(temporal_bin)
    taxi = pl.scan_parquet(GOLD_TAXI_PATH)

    if spatial_kind == "h3":
        if h3_resolution not in SUPPORTED_H3_RESOLUTIONS:
            raise ValueError(f"h3_resolution must be one of {SUPPORTED_H3_RESOLUTIONS}")
        spatial = make_h3_spatial_context(h3_resolution)
        taxi_col = f"pickup_h3_r{h3_resolution}"
        if h3_resolution == H3_RESOLUTION:
            taxi = taxi.with_columns(pl.col("pickup_h3_cell").alias(taxi_col))
        else:
            taxi = taxi.with_columns(h3_expr(
                "pickup_centroid_latitude", "pickup_centroid_longitude", h3_resolution, taxi_col
            ))
        skeleton = time_context.join(spatial, how="cross")
        result = add_taxi_data(skeleton, taxi, "h3_cell", taxi_col, temporal_bin=temporal_bin)
        stem = f"gold_demand_h3_r{h3_resolution}_{temporal_label}"
    elif spatial_kind == "census_tract":
        spatial = census_tracts_with_pois.with_columns(pl.col("census_tract").cast(pl.Int64))
        skeleton = time_context.join(spatial, how="cross")
        result = add_taxi_data(
            skeleton, taxi, "census_tract", "pickup_census_tract", temporal_bin=temporal_bin
        )
        stem = f"gold_demand_census_tract_{temporal_label}"
    else:
        raise ValueError("spatial_kind must be 'h3' or 'census_tract'")

    destination = output_path or (GOLD_TAXI_PATH.parent / f"{stem}.parquet")
    return result, destination


MODEL_DATASET_CONFIGS = [
    {"spatial_kind": "h3", "h3_resolution": res, "temporal_label": time_label}
    for res in SUPPORTED_H3_RESOLUTIONS for time_label in SUPPORTED_TEMPORAL_BINS
] + [
    {"spatial_kind": "census_tract", "h3_resolution": None, "temporal_label": time_label}
    for time_label in SUPPORTED_TEMPORAL_BINS
]

GENERATE_ALTERNATIVE_MODEL_DATASETS = False
if GENERATE_ALTERNATIVE_MODEL_DATASETS:
    for config in MODEL_DATASET_CONFIGS:
        dataset, destination = build_parameterized_demand(**config)
        print(f"Writing {destination.name} ...")
        dataset.sink_parquet(destination)
else:
    print("Parameterized builder ready. Set GENERATE_ALTERNATIVE_MODEL_DATASETS=True to materialize all configurations.")
    validation_rows = []
    for config in MODEL_DATASET_CONFIGS:
        dataset, destination = build_parameterized_demand(**config)
        schema = dataset.collect_schema()
        validation_rows.append({
            **config,
            "output_file": destination.name,
            "n_columns": len(schema),
            "has_target": "trip_count" in schema,
            "has_time": "datetime_hour" in schema,
        })
    builder_validation = pd.DataFrame(validation_rows)
    assert builder_validation[["has_target", "has_time"]].all().all()
    display(builder_validation)

Parameterized builder ready. Set GENERATE_ALTERNATIVE_MODEL_DATASETS=True to materialize all configurations.


,spatial_kind,h3_resolution,temporal_label,output_file,n_columns,has_target,has_time
0,h3,7.0,hourly,gold_demand_h3_r7_hourly.parquet,59,True,True
1,h3,7.0,4hourly,gold_demand_h3_r7_4hourly.parquet,59,True,True
2,h3,7.0,daily,gold_demand_h3_r7_daily.parquet,59,True,True
3,h3,8.0,hourly,gold_demand_h3_r8_hourly.parquet,59,True,True
4,h3,8.0,4hourly,gold_demand_h3_r8_4hourly.parquet,59,True,True
5,h3,8.0,daily,gold_demand_h3_r8_daily.parquet,59,True,True
6,h3,9.0,hourly,gold_demand_h3_r9_hourly.parquet,59,True,True
7,h3,9.0,4hourly,gold_demand_h3_r9_4hourly.parquet,59,True,True
8,h3,9.0,daily,gold_demand_h3_r9_daily.parquet,59,True,True
9,census_tract,NaN,hourly,gold_demand_census_tract_hourly.parquet,58,True,True


In [22]:
hexagon_hourly_demand = add_taxi_data(
    skeleton=hexagon_hourly,
    taxi=pl.scan_parquet(GOLD_TAXI_PATH),
    spatial_col="h3_cell",
    taxi_spatial_col="pickup_h3_cell",
)

hexagon_hourly_demand.sink_parquet(GOLD_HOURLY_DEMAND_HEXAGON)

community_area_hourly_demand = add_taxi_data(
    skeleton=community_area_hourly.with_columns(pl.col("community_area").cast(pl.Int64)),
    taxi=pl.scan_parquet(GOLD_TAXI_PATH),
    spatial_col="community_area",
    taxi_spatial_col="pickup_community_area",
)

community_area_hourly_demand.sink_parquet(GOLD_HOURLY_DEMAND_COMMUNITY_AREA)

census_tract_hourly_demand = add_taxi_data(
    skeleton=census_tract_hourly.with_columns(pl.col("census_tract").cast(pl.Int64)),
    taxi=pl.scan_parquet(GOLD_TAXI_PATH),
    spatial_col="census_tract",
    taxi_spatial_col="pickup_census_tract",
)

census_tract_hourly_demand.sink_parquet(GOLD_HOURLY_DEMAND_CENSUS_TRACT)

In [23]:
duckdb.sql(f"""
SELECT
    count(*)
FROM read_parquet('{GOLD_HOURLY_DEMAND_HEXAGON}')
""").show()

duckdb.sql(f"""
SELECT
    *
FROM read_parquet('{GOLD_HOURLY_DEMAND_HEXAGON}')
WHERE trip_count <> 0
LIMIT 10
""").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     17419966 │
└──────────────┘

┌─────────────────────┬───────┬─────────┬──────┬─────────────────────┬───────────────────────┬─────────────────────┬──────────────────────┬─────────────────────┬─────────────────────┬────────┬────────┬────────┬────────┬────────┬───────────┬───────────┬───────────┬───────────┬───────────┬───────────┬────────────┬────────────┬─────────────────┬────────────┬──────────┬────────┬───────────────┬────────────┬──────────────────┬────────────────────┬──────────────────┬──────────────────┬────────────────────┬────────────────────┬────────────────┬────────────────┬──────────┬───────────────────┬──────────┬──────────┬──────────┬───────────────────┬──────────┬──────────┬───────────┬────────────┬───────────┬───────────┬────────────┬─────────────┬────────────┬────────────┬────────────────────┬────────────────────┬────────────────┬────────────────┬──────────────────────────┐
│    datetime_hour    │ m

In [24]:
duckdb.sql(f"""
SELECT
    count(*)
FROM read_parquet('{GOLD_HOURLY_DEMAND_COMMUNITY_AREA}')
""").show()

duckdb.sql(f"""
SELECT
    *
FROM read_parquet('{GOLD_HOURLY_DEMAND_COMMUNITY_AREA}')
WHERE trip_count <> 0
LIMIT 10
""").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      1572494 │
└──────────────┘

┌─────────────────────┬───────┬─────────┬──────┬────────────────────┬───────────────────────┬─────────────────────┬──────────────────────┬────────────────────────┬──────────────────────┬────────┬────────┬────────┬────────┬────────┬───────────┬───────────┬───────────┬───────────┬───────────┬───────────┬────────────┬────────────┬────────────────┬────────────┬──────────┬────────┬───────────────┬────────────┬──────────────────┬───────────────────┬──────────────────┬──────────────────┬────────────────┬─────────────────┬────────────────┬────────────────┬──────────┬───────────┬──────────┬──────────┬──────────┬───────────┬──────────┬──────────┬───────────┬────────────┬───────────┬───────────┬────────────┬─────────────┬────────────┬────────────┬────────────────┬─────────────────┬────────────────┬────────────────┬──────────────────────────┐
│    datetime_hour    │ month │ weekday │ hour │     m

In [25]:
duckdb.sql(f"""
SELECT
    count(*)
FROM read_parquet('{GOLD_HOURLY_DEMAND_CENSUS_TRACT}')
""").show()

duckdb.sql(f"""
SELECT
    *
FROM read_parquet('{GOLD_HOURLY_DEMAND_CENSUS_TRACT}')
WHERE trip_count <> 0
LIMIT 10
""").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     17930516 │
└──────────────┘

┌─────────────────────┬───────┬─────────┬──────┬────────────────────────┬─────────────────────────┬─────────────────────┬──────────────────────┬────────────────────────┬───────────────────────┬────────┬────────┬────────┬────────┬────────┬───────────┬───────────┬───────────┬───────────┬───────────┬───────────┬────────────┬────────────┬──────────────┬────────────┬──────────┬────────┬───────────────┬────────────┬──────────────────┬───────────────────┬──────────────────┬──────────────────┬────────────────────┬───────────────────┬────────────────┬────────────────┬──────────┬───────────┬──────────┬──────────┬──────────┬───────────┬──────────┬──────────┬───────────┬────────────┬───────────┬───────────┬────────────┬─────────────┬────────────┬────────────┬────────────────┬─────────────────┬────────────────┬────────────────┬──────────────────────────┐
│    datetime_hour    │ month │ weekday │ h

## Quality Check

In [26]:
con = duckdb.connect()

columns = con.sql(f"""
DESCRIBE SELECT *
FROM read_parquet('{GOLD_HOURLY_DEMAND_COMMUNITY_AREA}')
""").df()["column_name"].tolist()

parts = []

for col in columns:
    parts.append(f"""
    SELECT
        '{col}' AS column_name,
        COUNT(*) AS n_rows,
        SUM(CASE WHEN "{col}" IS NULL THEN 1 ELSE 0 END) AS n_nulls,
        ROUND(
            100.0 * SUM(CASE WHEN "{col}" IS NULL THEN 1 ELSE 0 END) / COUNT(*),
            2
        ) AS null_pct
    FROM read_parquet('{GOLD_HOURLY_DEMAND_COMMUNITY_AREA}')
    """)

query = "\nUNION ALL\n".join(parts) + "\nORDER BY null_pct DESC"

null_report = con.sql(query).df()
print(null_report.to_string(index=False))

             column_name  n_rows  n_nulls  null_pct
                   month 1572494      0.0       0.0
              extras_sum 1572494      0.0       0.0
                hour_cos 1572494      0.0       0.0
                    relh 1572494      0.0       0.0
           datetime_hour 1572494      0.0       0.0
                 weekday 1572494      0.0       0.0
                    tmpc 1572494      0.0       0.0
               skyc1_BKN 1572494      0.0       0.0
                    vsby 1572494      0.0       0.0
                    sknt 1572494      0.0       0.0
               skyc1_CLR 1572494      0.0       0.0
               skyc1_OVC 1572494      0.0       0.0
               skyc1_FEW 1572494      0.0       0.0
                    p01m 1572494      0.0       0.0
              is_holiday 1572494      0.0       0.0
                hour_sin 1572494      0.0       0.0
               skyc1_SCT 1572494      0.0       0.0
               skyc1_VV  1572494      0.0       0.0
            